In [ ]:

import csv
import json

# Read CSV file
with open("/content/appointments.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    headers = reader.fieldnames
    rows = list(reader)

print("Headers:", headers)
print("Total Rows:", len(rows))


Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows: 1020


In [ ]:

def parse_row(row):

    converted = row.copy()

    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])

    return converted


In [ ]:

accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted Rows:", len(accepted))
print("Rejected Rows:", len(rejected))


Accepted Rows: 988
Rejected Rows: 32


In [ ]:

with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected data saved successfully!")


Rejected data saved successfully!


In [ ]:
print("Accepted Rows:", len(accepted))
print("Rejected Rows:", len(rejected))

Accepted Rows: 988
Rejected Rows: 32


**STEP 1.2 — Pandas Data Cleaning (Pass 1)**

In [ ]:

import pandas as pd
import numpy as np


In [ ]:

df = pd.read_csv("/content/appointments.csv")

print("Dataset Shape:", df.shape)


Dataset Shape: (1020, 14)


In [ ]:

print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id          0
clinic_id 

In [ ]:

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [ ]:

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].value_counts())
print("\nUnique Channels:")
print(df["booking_channel"].unique())


booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64

Unique Channels:
['Phone' 'Walk-In' 'App']


In [ ]:

df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

print("Date conversion completed.")


Date conversion completed.


In [ ]:

df = df.dropna(
    subset=["appt_date", "status"]
)

print("Rows after removing missing dates and status:", len(df))


Rows after removing missing dates and status: 990


In [ ]:

print("Final Dataset Shape:", df.shape)

print("\nRemaining Missing Values:")
print(df[["appt_date", "status"]].isnull().sum())


Final Dataset Shape: (990, 14)

Remaining Missing Values:
appt_date    0
status       0
dtype: int64


**STEP 1.3 — Data Cleaning (Pass 2)**

In [ ]:

df.loc[
    (df["age"] < 1) | (df["age"] > 100),
    "age"
] = np.nan

print("Invalid ages:", df["age"].isnull().sum())


Invalid ages: 17


In [ ]:

# Fill missing age with median
age_median = df["age"].median()

df["age"] = df["age"].fillna(
    age_median
).round().astype(int)


# Fill missing distance with median
distance_median = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

print("Age Median:", age_median)
print("Distance Median:", distance_median)

print("\nMissing Values:")
print(df[["age", "distance_km"]].isnull().sum())


Age Median: 48.0
Distance Median: 8.25

Missing Values:
age            0
distance_km    0
dtype: int64


In [ ]:

# Distance Outlier Calculation

Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - (1.5 * IQR_distance)
upper_distance = Q3_distance + (1.5 * IQR_distance)


# Fee Outlier Calculation

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - (1.5 * IQR_fee)
upper_fee = Q3_fee + (1.5 * IQR_fee)


print("Distance Q1:", Q1_distance)
print("Distance Q3:", Q3_distance)
print("Distance IQR:", IQR_distance)
print("Distance Lower Fence:", lower_distance)
print("Distance Upper Fence:", upper_distance)

print("\nFee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)
print("Fee Lower Fence:", lower_fee)
print("Fee Upper Fence:", upper_fee)


Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Distance Lower Fence: -7.9375
Distance Upper Fence: 25.5625

Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee Lower Fence: 68.75
Fee Upper Fence: 1218.75


In [ ]:

# Count Distance Outliers
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()


# Count Fee Outliers
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()


# Apply np.clip
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)


print("Distance Values Capped:", distance_capped)
print("Fee Values Capped:", fee_capped)


Distance Values Capped: 26
Fee Values Capped: 3


In [ ]:

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df[["distance_km", "fee_inr"]].head(10))


    distance_km  fee_inr
0          18.0      850
1          12.5      350
2           4.1      850
4           7.5      750
5          19.0      700
6          11.6      400
7          14.0      300
8           6.1      850
9           3.0      300
10          5.0      700


In [ ]:

print("Final Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nMaximum Distance:", df["distance_km"].max())
print("Maximum Fee:", df["fee_inr"].max())


Final Dataset Shape: (990, 14)

Missing Values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Maximum Distance: 25.6
Maximum Fee: 1219


**STEP 1.4 — Feature Engineering & NumPy Analysis**

In [ ]:

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print("Minimum Lead Days:", df["lead_days"].min())
print("Maximum Lead Days:", df["lead_days"].max())


Minimum Lead Days: 1
Maximum Lead Days: 45


In [ ]:

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

print(df["no_show"].value_counts())


no_show
0    772
1    218
Name: count, dtype: int64


In [ ]:

X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print("NumPy Array Shape:", X.shape)
print("\nFirst 5 Rows:")
print(X[:5])


NumPy Array Shape: (990, 4)

First 5 Rows:
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 [ 52.    7.5 750.    4. ]
 [ 64.   19.  700.   40. ]]


In [ ]:

column_means = np.mean(X, axis=0)

print("Column Means:", column_means)


Column Means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:

mask = df["lead_days"] > 14

count = mask.sum()
share = mask.mean()

print("Appointments above 14 days:", count)
print("Share:", round(share * 100, 1), "%")


Appointments above 14 days: 296
Share: 29.9 %


In [ ]:

distance = X[:, 1]

z_scores = (
    distance - np.mean(distance)
) / np.std(distance)

outlier_count = np.sum(np.abs(z_scores) > 2)

print("Distance Z-Score Outliers:", outlier_count)


Distance Z-Score Outliers: 53


In [ ]:

print("Total Appointments:", len(df))

print("Total No-Shows:", df["no_show"].sum())

print(
    "No-Show Percentage:",
    round(df["no_show"].mean() * 100, 1),
    "%"
)

print("Lead Days Range:",
      df["lead_days"].min(),
      "to",
      df["lead_days"].max())


Total Appointments: 990
Total No-Shows: 218
No-Show Percentage: 22.0 %
Lead Days Range: 1 to 45


**STEP 1.5 Enrich Clinics Using Live API**

In [ ]:

import requests
import pandas as pd


In [ ]:

clinics = pd.read_csv("/content/clinics.csv")

print("Total Clinics:", len(clinics))
print("\nClinic Information:")
print(clinics.head())


Total Clinics: 8

Clinic Information:
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873


In [ ]:

def get_temperature(latitude, longitude):

    for attempt in range(2):

        try:
            url = "https://api.open-meteo.com/v1/forecast"

            params = {
                "latitude": latitude,
                "longitude": longitude,
                "hourly": "temperature_2m",
                "forecast_days": 1
            }

            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP Status: {response.status_code}"
                continue

            data = response.json()

            temperatures = data.get(
                "hourly", {}
            ).get("temperature_2m")

            if not temperatures:
                reason = "Temperature data missing"
                continue

            return temperatures[0]

        except requests.exceptions.RequestException as e:
            reason = str(e)

        except (ValueError, KeyError, TypeError, IndexError) as e:
            reason = str(e)

    return None


In [ ]:

temperatures = []

for index, row in clinics.iterrows():

    temp = get_temperature(
        row["latitude"],
        row["longitude"]
    )

    temperatures.append(temp)

    if temp is None:
        print(
            "Failed:",
            row["clinic_name"],
            "- Temperature unavailable"
        )
    else:
        print(
            row["clinic_name"],
            "Temperature:",
            temp,
            "°C"
        )

clinics["temp_c"] = temperatures


Kothrud Family Clinic Temperature: 17.1 °C
Hadapsar Care Centre Temperature: 18.2 °C
Sitabuldi Health Point Temperature: 20.7 °C
Malviya Nagar Clinic Temperature: 17.1 °C
Vaishali Nagar Clinic Temperature: 17.1 °C
Edappally Medical Centre Temperature: 24.7 °C
Vijay Nagar Clinic Temperature: 20.3 °C
Gomti Nagar Care Point Temperature: 15.7 °C


In [ ]:

clinics.to_csv(
    "clinics_enriched.csv",
    index=False
)

print("Clinics Enriched Dataset Saved!")


Clinics Enriched Dataset Saved!


In [ ]:

print("Dataset Shape:", clinics.shape)

print("\nEnriched Dataset:")
print(clinics)

print("\nTemperature Information:")
print(clinics["temp_c"].describe())

print("\nMissing Temperatures:")
print(clinics["temp_c"].isnull().sum())


Dataset Shape: (8, 7)

Enriched Dataset:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.1  
1    18.2  
2    20.7  
3    17.1  
4    17.1  
5    24.7  
6    20.3  
7    15.7  

Temperature Information:
count     8.00000
mean     18.86250
std       2.91005
min      15.70000
25%      17.10000
50%      17.65000
75%  

**STEP 1.6 — Final Data Export & SQL File Creation**

In [ ]:

df = df.sort_values(
    by="appointment_id"
).reset_index(drop=True)

print("Sorted Dataset:")
print(df[["appointment_id", "patient_id"]].head(10))


Sorted Dataset:
  appointment_id patient_id
0          A0001       P509
1          A0002       P046
2          A0003       P009
3          A0004       P482
4          A0005       P251
5          A0006       P280
6          A0007       P349
7          A0008       P445
8          A0009       P383
9          A0010       P358


In [ ]:

df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")

df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

print(df[["booked_on", "appt_date"]].head())


    booked_on   appt_date
0  2025-01-24  2025-01-25
1  2025-05-27  2025-06-06
2  2025-03-24  2025-03-30
3  2025-04-18  2025-04-22
4  2024-11-23  2025-01-02


In [ ]:

columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

df_clean = df[columns].copy()

print("Clean Dataset Shape:", df_clean.shape)
print("\nColumns:")
print(df_clean.columns.tolist())


Clean Dataset Shape: (990, 16)

Columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:

print("Unique Appointment IDs:",
      df_clean["appointment_id"].nunique())

print("Total Rows:", len(df_clean))

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nTotal Missing Values:",
      df_clean.isnull().sum().sum())


Unique Appointment IDs: 990
Total Rows: 990

Missing Values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total Missing Values: 0


In [ ]:

df_clean.to_csv(
    "appointments_clean.csv",
    index=False
)

print("appointments_clean.csv saved successfully!")


appointments_clean.csv saved successfully!


In [ ]:

def sql_value(value):

    if pd.isna(value):
        return "NULL"

    if isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)

    return "'" + str(value) + "'"


In [ ]:

sql_statements = []

for _, row in clinics.iterrows():

    values = [
        sql_value(row["clinic_id"]),
        sql_value(row["clinic_name"]),
        sql_value(row["city"]),
        sql_value(row["tier"]),
        sql_value(row["latitude"]),
        sql_value(row["longitude"]),
        sql_value(row["temp_c"])
    ]

    statement = (
        f"INSERT INTO clinics VALUES ({','.join(values)});"
    )

    sql_statements.append(statement)

print("Clinic INSERT Statements:",
      len(sql_statements))


Clinic INSERT Statements: 8


In [ ]:

for _, row in df_clean.iterrows():

    values = [sql_value(value) for value in row]

    statement = (
        f"INSERT INTO appointments VALUES ({','.join(values)});"
    )

    sql_statements.append(statement)

print("Total SQL INSERT Statements:",
      len(sql_statements))


Total SQL INSERT Statements: 998


In [ ]:

with open("load_data.sql", "w") as file:

    for statement in sql_statements:
        file.write(statement + "\n")

print("load_data.sql created successfully!")


load_data.sql created successfully!


In [ ]:

print("Appointments:", len(df_clean))
print("Clinics:", len(clinics))
print("SQL Statements:", len(sql_statements))

print("\nFinal Files:")

import os

for filename in [
    "appointments_clean.csv",
    "clinics_enriched.csv",
    "load_data.sql"
]:
    print(filename, "Exists:", os.path.exists(filename))


Appointments: 990
Clinics: 8
SQL Statements: 998

Final Files:
appointments_clean.csv Exists: True
clinics_enriched.csv Exists: True
load_data.sql Exists: True


In [ ]:

# ============================================
# CAPSTONE PROJECT 1
# COLAB TO GOOGLE DRIVE FILE TRANSFER
# ============================================

from google.colab import drive
from pathlib import Path
import shutil

# STEP 1: Connect Google Drive
drive.mount('/content/drive')

# STEP 2: Destination Folder
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAPSTONE_PROJECT_1"
)

DATA_DIR = PROJECT_DIR / "data"

# Create folders if not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# STEP 3: Files to Transfer
files_to_copy = [
    "appointments.csv",
    "appointments_clean.csv",
    "clinics.csv",
    "clinics_enriched.csv",
    "followup.csv",
    "load_data.sql",
    "rejected_rows.json"
]

# STEP 4: Copy Files
for filename in files_to_copy:

    source = Path("/content") / filename
    destination = DATA_DIR / filename

    if source.exists() and source.is_file():

        shutil.copy2(source, destination)

        print("COPIED:", filename)

    else:
        print("NOT FOUND:", filename)

# STEP 5: Copy Existing Data Folder
source_data = Path("/content/data")

if source_data.exists():

    for item in source_data.iterdir():

        destination = DATA_DIR / item.name

        if item.is_dir():
            shutil.copytree(
                item,
                destination,
                dirs_exist_ok=True
            )
        else:
            shutil.copy2(item, destination)

    print("Existing data folder copied!")

# STEP 6: Verify Files
print("\n========== TRANSFER COMPLETE ==========")

for item in sorted(DATA_DIR.iterdir()):
    print(item.name)

print("\nGoogle Drive Location:")
print(DATA_DIR)


Mounted at /content/drive
COPIED: appointments.csv
COPIED: appointments_clean.csv
COPIED: clinics.csv
COPIED: clinics_enriched.csv
COPIED: followup.csv
COPIED: load_data.sql
COPIED: rejected_rows.json

========== TRANSFER COMPLETE ==========
appointments.csv
appointments_clean.csv
clinics.csv
clinics_enriched.csv
followup.csv
followup_notes.csv
load_data.sql
rejected_rows.json

Google Drive Location:
/content/drive/MyDrive/CAPSTONE_PROJECT_1/data


**PART 2 — SQL Analysis**
                          **STEP 2.1 — Build and Load Database**


In [ ]:

from google.colab import files

files.download("load_data.sql")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

clinic_analysis = df_clean.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="left"
)

clinic_summary = clinic_analysis.groupby(
    ["clinic_name", "city"]
).agg(
    appointments=("no_show", "count"),
    no_shows=("no_show", "sum")
).reset_index()

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] /
    clinic_summary["appointments"]
).round(1)

clinic_summary = clinic_summary.sort_values(
    "no_show_pct",
    ascending=False
)

print(clinic_summary)


                clinic_name     city  appointments  no_shows  no_show_pct
1    Gomti Nagar Care Point  Lucknow            90        30         33.3
2      Hadapsar Care Centre     Pune           143        39         27.3
4      Malviya Nagar Clinic   Jaipur           142        38         26.8
3     Kothrud Family Clinic     Pune           160        36         22.5
7        Vijay Nagar Clinic   Indore           101        21         20.8
0  Edappally Medical Centre    Kochi           142        23         16.2
5    Sitabuldi Health Point   Nagpur            82        13         15.9
6     Vaishali Nagar Clinic   Jaipur           130        18         13.8


In [ ]:

print(
    "Total Appointments:",
    clinic_summary["appointments"].sum()
)

print(
    "Total No-Shows:",
    clinic_summary["no_shows"].sum()
)


Total Appointments: 990
Total No-Shows: 218
